%md
# **Spark Project - KPIs Notebook**

In [0]:
order_details_saved = spark.table("order_details_curated")

order_details_saved.show(5)

+--------+-------+----------+--------------------+--------------------+------------+------------+---------+-----------------+----------------------+-----------------+---------+
|order_id|user_id|product_id|        product_name|               aisle|  department|order_number|order_dow|order_hour_of_day|days_since_prior_order|add_to_cart_order|reordered|
+--------+-------+----------+--------------------+--------------------+------------+------------+---------+-----------------+----------------------+-----------------+---------+
|     192| 189954|      5876|       Organic Lemon|        fresh fruits|     produce|          44|        5|               14|                   8.0|                1|        1|
|     192| 189954|     49476| Organic Maple Syrup|honeys syrups nec...|      pantry|          44|        5|               14|                   8.0|                2|        1|
|     192| 189954|      9598|Gogo Squeez Organ...|canned fruit appl...|canned goods|          44|        5|        

**Part 3: Build a small KPI (Key Performance Indicator) layer**

In [0]:
from pyspark.sql.functions import count, col

dept_kpi = (
    order_details_saved
    .groupBy("department")
    .agg(count("*").alias("total_items"))
    .orderBy(col("total_items").desc())
)

dept_kpi.show(10)

+---------------+-----------+
|     department|total_items|
+---------------+-----------+
|        produce|    9870199|
|     dairy eggs|    5631067|
|         snacks|    3006412|
|      beverages|    2804175|
|         frozen|    2336858|
|         pantry|    1956819|
|         bakery|    1225181|
|   canned goods|    1114857|
|           deli|    1095540|
|dry goods pasta|     905340|
+---------------+-----------+
only showing top 10 rows


This KPI shows which departments have the highest number of items ordered. This, in turn, helps a business leader identify which departments are needed to have the largest inventory turnover.

In [0]:
from pyspark.sql.functions import count

reorder_kpi = (
    order_details_saved
    .filter("reordered = 1")
    .groupBy("order_hour_of_day")
    .agg(count("*").alias("reorder_count"))
    .orderBy(col("reorder_count").desc())
)

reorder_kpi.show(10)

+-----------------+-------------+
|order_hour_of_day|reorder_count|
+-----------------+-------------+
|               10|      1726499|
|               11|      1676271|
|               14|      1635288|
|               15|      1615183|
|               13|      1612121|
|               12|      1583898|
|                9|      1580305|
|               16|      1533153|
|               17|      1258378|
|                8|      1130174|
+-----------------+-------------+
only showing top 10 rows


This KPI shows when customers are most likely to reorder items. This can help business leaders know the best time to send alerts and notifications to customers as reminders to reorder more items.

In [0]:
from pyspark.sql.functions import *

basket_kpi = (
    order_details_saved
    .groupBy("order_id")
    .agg(count("*").alias("items_in_order"))
)

basket_size_table = (
    basket_kpi
    .withColumn("basket_size_bin",
        when(col("items_in_order") == 1, "1 item")
        .when((col("items_in_order") >= 2) & (col("items_in_order") <= 3), "2-3 items")
        .when((col("items_in_order") >= 4) & (col("items_in_order") <= 5), "4-5 items")
        .when((col("items_in_order") >= 6) & (col("items_in_order") <= 8), "6-8 items")
        .when((col("items_in_order") >= 9) & (col("items_in_order") <= 12), "9-12 items")
        .when(col("items_in_order") >= 13, "13+ items")
        .otherwise("Other")
    )
    .withColumn("bin_order",
        when(col("basket_size_bin") == "1 item", 1)
        .when(col("basket_size_bin") == "2-3 items", 2)
        .when(col("basket_size_bin") == "4-5 items", 3)
        .when(col("basket_size_bin") == "6-8 items", 4)
        .when(col("basket_size_bin") == "9-12 items", 5)
        .when(col("basket_size_bin") == "13+ items", 6)
        .otherwise(7)
    )
    .groupBy("basket_size_bin", "bin_order")
    .agg(
        count("*").alias("number_of_orders"),
        round(avg("items_in_order"), 2).alias("avg_items_per_order_in_bin"),
        round((count("*") / basket_kpi.count()) * 100, 2).alias("percentage_of_orders")
    )
    .orderBy("bin_order")
    .drop("bin_order")
)

basket_size_table.show(truncate=False)

+---------------+----------------+--------------------------+--------------------+
|basket_size_bin|number_of_orders|avg_items_per_order_in_bin|percentage_of_orders|
+---------------+----------------+--------------------------+--------------------+
|1 item         |163593          |1.0                       |4.89                |
|2-3 items      |409421          |2.53                      |12.24               |
|4-5 items      |467524          |4.51                      |13.97               |
|6-8 items      |676287          |6.96                      |20.21               |
|9-12 items     |654125          |10.36                     |19.55               |
|13+ items      |975133          |19.51                     |29.14               |
+---------------+----------------+--------------------------+--------------------+



I got a bit carried away with this one... This KPI is used to "bin" the size of the customer orders to measure how many items are in a typical order. From this, it seems that the 13+ bin is the largest with 29% of all orders falling in this bin. This suggests many users will get lots of items per order, but still half of all orders being under 8 items. This help business leaders coordinate pricing strategies or bundles that may increase more items being purchased. 

**Part 4: Advanced analysis challenge**

In [0]:
from pyspark.sql.functions import col, count, max, when, lit, round, desc, asc
 
# splitting the data
max_order_id = order_details_saved.select(max("order_id")).collect()[0][0]
split_point = max_order_id / 2

first_half = order_details_saved.filter(col("order_id") <= split_point)
second_half = order_details_saved.filter(col("order_id") > split_point)

# counting the number of products in each group.
first_counts = (first_half
    .groupBy("product_id", "product_name")
    .agg(count("*").alias("count_first"))
)

second_counts = (second_half
    .groupBy("product_id", "product_name")
    .agg(count("*").alias("count_second"))
)

# combining all metrics into one table
product_trends = (first_counts
    .join(second_counts, on=["product_id", "product_name"], how="full_outer")
    .withColumn("count_first", when(col("count_first").isNull(), 0).otherwise(col("count_first")))
    .withColumn("count_second", when(col("count_second").isNull(), 0).otherwise(col("count_second")))
    .withColumn("absolute_change", col("count_second") - col("count_first"))
    .withColumn("percentage_change", 
                when(col("count_first") > 0, 
                     round((col("count_second") - col("count_first")) / col("count_first") * 100, 1))
                .otherwise(lit(None)))
    .filter(col("product_name").isNotNull())
    .orderBy(desc("absolute_change"))
)

# showing the business insight
print("=== TOP 10 GAINERS (Biggest Absolute Increase) ===")
product_trends.select(
    "product_name", "count_first", "count_second", 
    "absolute_change", "percentage_change"
).limit(10).show(truncate=False)

print("=== TOP 10 DECLINERS (Biggest Absolute Decrease) ===")
product_trends.orderBy(asc("absolute_change")).select(
    "product_name", "count_first", "count_second", 
    "absolute_change", "percentage_change"
).limit(10).show(truncate=False)

=== TOP 10 GAINERS (Biggest Absolute Increase) ===
+--------------------------+-----------+------------+---------------+-----------------+
|product_name              |count_first|count_second|absolute_change|percentage_change|
+--------------------------+-----------+------------+---------------+-----------------+
|Organic Baby Spinach      |125492     |126213      |721            |0.6              |
|Asparagus                 |35257      |35894       |637            |1.8              |
|Organic Raspberries       |71002      |71601       |599            |0.8              |
|Extra Virgin Olive Oil    |25882      |26441       |559            |2.2              |
|Organic Cucumber          |42270      |42735       |465            |1.1              |
|Organic Avocado           |91881      |92343       |462            |0.5              |
|100% Raw Coconut Water    |19421      |19850       |429            |2.2              |
|Organic Kiwi              |25798      |26223       |425            |

This query is my attempt to show which products change the most over time. Assuming that the order_id is generated in chronological order,  I split the data into two equal halves, and compared theier change in order occurance. Finally, I used a full outer join, which is unlike the majority of examples done in class, because I did not want to lose products that disappeared in the second half, or that just only had records in the second half and not first. However, from a rough look at the data, it does not seem like this dataset would have been effected if I had chosen to do different type of join.

In my first pass, I saw that the leaders were "NULL" values, so I added a .isNotNull() filter. 

The next dilema I ran into was whether I should sort on absolute change or percent change. I chose to sort primarily by absolute change because it reflects the true business impact on revenue and operations. Percentage change is still shown as a secondary column to help identify fast-moving emerging products.

Finally, I wanted to show the top 10 gainers and decliners, because that is truly the metric that business leaders are looking at to identify what products have the largest fluxuation of volume. Since this dataset does not have a cost associated with the orders or products, this is the next best solution. 

**Part 5: Engineering reflection**

1. Why does one row in order_details represent one product within one order rather than one order?

_This is because most business questions are driven at the product-level. If I collapsed the data to one row per order, I would lose the ability to analyze individual product behavior._

2. In your joins, where might a left join be preferable to an inner join, and why?

_A left join was preferrable when merging products, aisles, and departments to the order_products table because an inner join would have dropped any order rows where the product data was missing._

3. Why might a team choose to build a reusable curated table first instead of writing a separate pipeline for every business question?

_By building a resulable table, every future KPI that a business leader wants to perform does not have to repeat the complex join queries necessary to colocate the data. Instead, having a resuable table reduces computation time and potential errors made by rebuilding the table._

4. Suppose we want to answer the question: Which products are more commonly bought on weekdays than on weekends?
Why would rows from orders with no matching order_products row be unhelpful in order_details for this analysis?

_Orders with no matching order_products rows do not provide any product information, so they would just add noise to the data and make any aggregations less accurate._

5. Why is it useful to keep valid order-item rows in order_details even when some aisle or department information is missing, rather than dropping those rows entirely?

_Even if the aisle or department information is NULL, they still provide valuable business data. If these rows were to be dropped, it would under count the total volume of sales and bias KPIs in an unintentional manner. Since aisle and departments are purely product metadata, it is important to keep them in the dataset with an "unknown" value instead of removing them._